In [1]:
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

# Import RAG

In [2]:
import sys
sys.path.append('../scripts')
import rag
import vectors

# Loading data

## Answers to synthetic questions

In [3]:
df_synth_a = pd.read_csv('../data/data-synth-answer.csv', sep='\t', dtype=str)
df_synth_a

,pmid,ollama_seed,answer_llama3.2:1b,answer_gemma3:1b
0,40247149,0,Based on the provided context and papers from ...,"Okay, here's an answer based on the provided t..."
1,40247149,1,Based on the context provided by the papers fr...,"Okay, let’s tackle this question based on the ..."
2,40247149,2,Researchers believe that software tools play a...,"Okay, based on the provided context and papers..."
3,40247149,3,Based on the context provided by the papers yo...,"Okay, let’s analyze this context and answer yo..."
4,40247149,4,One potential limitation of using functional m...,"Okay, here’s an analysis of the provided text,..."
5,40266512,0,Based on the provided context and PubMed paper...,"Okay, based on the provided context and the pr..."
6,40266512,1,Based on the context provided by the papers fr...,"Okay, here's an analysis of the provided text,..."
7,40266512,2,Based on the provided context and papers from ...,"Okay, let’s analyze the provided papers concer..."
8,40266512,3,Based on the provided context and papers from ...,"Okay, here's an answer based on the provided c..."
9,40266512,4,Based on the context provided by the papers fr...,"Okay, let’s analyze the provided papers focusi..."


## Synthetic questions

In [4]:
df_synth_q = pd.read_csv('../data/data-synth-question.csv', sep='\t', dtype=str)
df_synth_q

,pmid,ollama_seed,synthetic_question
0,40247149,0,What role do computer algorithms play in devel...
1,40247149,1,Is it more efficient to use machine learning a...
2,40247149,2,What role do researchers believe software tool...
3,40247149,3,What role do expert systems play in improving ...
4,40247149,4,What is one potential limitation of using fMRI...
...,...,...,...
495,40933682,0,What is a major challenge that autistic studen...
496,40933682,1,Does a lack of visibility and understanding fr...
497,40933682,2,What drives college students with autism to de...
498,40933682,3,What is the primary reason that autistic colle...


## Abstracts

In [5]:
kb_records = rag.get_kb_records('../data/data-kb.csv')
df_abstr = pd.DataFrame.from_records(kb_records)[['pmid', 'abstract']]
df_abstr

,pmid,abstract
0,40938348,
1,40938167,Stereotypies currently occupy an important pla...
2,40938164,The essay delineates a psychic phenomenon of i...
3,40938123,"Here, we report the draft genome sequence of"
4,40937558,Adolescence is a time of complex social and em...
...,...,...
2995,40245385,An individual education plan (IEP) is a key el...
2996,40244830,
2997,40244560,Retinal ganglion cells (RGCs) are the only neu...
2998,40244507,The aim is to examine the relationship between...


## Put together

In [6]:
df_together = df_synth_a.copy()
df_together = df_together.merge(df_synth_q, on=['pmid', 'ollama_seed'], how='left')
df_together = df_together.merge(df_abstr, on=['pmid'], how='left')
df_together

,pmid,ollama_seed,answer_llama3.2:1b,answer_gemma3:1b,synthetic_question,abstract
0,40247149,0,Based on the provided context and papers from ...,"Okay, here's an answer based on the provided t...",What role do computer algorithms play in devel...,Functional magnetic resonance imaging (fMRI) h...
1,40247149,1,Based on the context provided by the papers fr...,"Okay, let’s tackle this question based on the ...",Is it more efficient to use machine learning a...,Functional magnetic resonance imaging (fMRI) h...
2,40247149,2,Researchers believe that software tools play a...,"Okay, based on the provided context and papers...",What role do researchers believe software tool...,Functional magnetic resonance imaging (fMRI) h...
3,40247149,3,Based on the context provided by the papers yo...,"Okay, let’s analyze this context and answer yo...",What role do expert systems play in improving ...,Functional magnetic resonance imaging (fMRI) h...
4,40247149,4,One potential limitation of using functional m...,"Okay, here’s an analysis of the provided text,...",What is one potential limitation of using fMRI...,Functional magnetic resonance imaging (fMRI) h...
5,40266512,0,Based on the provided context and PubMed paper...,"Okay, based on the provided context and the pr...",What are the differences between the brain fun...,Cognitive dysmetria suggests a disorganization...
6,40266512,1,Based on the context provided by the papers fr...,"Okay, here's an analysis of the provided text,...",Here's a general question about autism that ca...,Cognitive dysmetria suggests a disorganization...
7,40266512,2,Based on the provided context and papers from ...,"Okay, let’s analyze the provided papers concer...",What are the key differences between autism sp...,Cognitive dysmetria suggests a disorganization...
8,40266512,3,Based on the provided context and papers from ...,"Okay, here's an answer based on the provided c...",What are some common characteristics or differ...,Cognitive dysmetria suggests a disorganization...
9,40266512,4,Based on the context provided by the papers fr...,"Okay, let’s analyze the provided papers focusi...",How do cognitive dysmetria and functional conn...,Cognitive dysmetria suggests a disorganization...


# Evaluation by cosine similarity

## Embed texts

In [7]:
# vectorizer handle
print(vectors.model_handle)

multi-qa-MiniLM-L6-cos-v1


In [8]:
# vector representations
print(datetime.now())
column_names_to_vectorize = ['answer_llama3.2:1b', 'answer_gemma3:1b', 'abstract']
vectorized = {name : vectors.model.encode(df_together[name].to_list()) \
for name in column_names_to_vectorize}
print(datetime.now())

2025-09-13 00:44:01.117272
2025-09-13 00:44:06.024778


## Compute cosine similarities

In [9]:
def get_cosine_similarities(model_handle):
    similarities = []
    for i in range(len(vectorized['abstract'])):
        vector_answer = vectorized['answer_'+model_handle][i]
        vector_abstract = vectorized['abstract'][i]
        similarity = vectors.model.similarity(vector_answer, vector_abstract).item()
        similarities.append(similarity)
    return similarities

In [10]:
# get similarities by model handle
df_similarities = pd.DataFrame({handle : get_cosine_similarities(handle) \
for handle in ['llama3.2:1b', 'gemma3:1b']})
df_similarities

,llama3.2:1b,gemma3:1b
0,0.616656,0.563908
1,0.594769,0.562691
2,0.679445,0.606250
3,0.446740,0.573610
4,0.785160,0.552165
5,0.479278,0.506215
6,0.185969,0.260058
7,0.550898,0.521722
8,0.477883,0.426746
9,0.811491,0.770920


## Compare models

In [11]:
# descriptive statistics, for llama3.2
df_similarities['llama3.2:1b'].describe()

count    30.000000
mean      0.657572
std       0.162292
min       0.185969
25%       0.620560
50%       0.701615
75%       0.762441
max       0.915637
Name: llama3.2:1b, dtype: float64

In [12]:
# descriptive statistics, for llama3.2
df_similarities['gemma3:1b'].describe()

count    30.000000
mean      0.612760
std       0.130359
min       0.260058
25%       0.562995
50%       0.634651
75%       0.691649
max       0.851179
Name: gemma3:1b, dtype: float64

In [13]:
# descriptive statistics, difference
(df_similarities['gemma3:1b']-df_similarities['llama3.2:1b']).describe()

count    30.000000
mean     -0.044811
std       0.073026
min      -0.232995
25%      -0.072283
50%      -0.051915
75%      -0.024539
max       0.126870
dtype: float64

In [14]:
print(datetime.now())

2025-09-13 00:44:06.074999
